In [1]:
import numpy as np
import pandas as pd

### Read the Dataset
Dataset: data/car_fuel_efficiency_2026.csv

In [2]:
df = pd.read_csv("data/car_fuel_efficiency_2026.csv")
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


### Q1. Pandas version  
What version of Pandas did you install?

In [3]:
print("Numpy Version: ", np.__version__)
print("Pandas Version: ", pd.__version__)

Numpy Version:  2.5.3
Pandas Version:  3.0.5


### Q2. Records count  
How many records are in the dataset?

In [4]:
print("Number of records:", df.shape[0])  # shape gives (rows, columns)

Number of records: 10000


### Q3. Fuel types  
How many fuel types are presented in the dataset?

In [5]:
print(df["fuel_type"].unique())
print()
print("Number of fuel types:", df["fuel_type"].nunique())

<StringArray>
['Gasoline', 'Diesel', 'Hybrid']
Length: 3, dtype: str

Number of fuel types: 3


### Q4. Missing values  
How many columns in the dataset have missing values?

In [6]:
print(df.isnull().sum())
print()
print("Columns with missing values:", (df.isnull().sum() > 0).sum())

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

Columns with missing values: 2


### Q5. Max fuel efficiency  
What's the maximum fuel efficiency of cars from Asia?

In [7]:
print(
    "Max efficiency:",
    df[df["origin"].str.lower() == "asia"]["fuel_efficiency_mpg"].max(),
)

Max efficiency: 41.2


### Q6. Median value of horsepower
- Find the median value of the horsepower column in the dataset.
- Next, calculate the most frequent value of the same horsepower column.
- Use the fillna method to fill the missing values in the horsepower column with the most frequent value from the previous step.
- Now, calculate the median value of horsepower once again.


In [8]:
# Median & Mode
print("Median horsepower:", df["horsepower"].median())
print("Most frequent horsepower:", df["horsepower"].mode()[0])

Median horsepower: 254.0
Most frequent horsepower: 252.0


In [9]:
# Fill missing horsepower values with the most frequent value
most_frequent = df["horsepower"].mode()[0]
df["horsepower"] = df["horsepower"].fillna(most_frequent)

In [10]:
# Median of horsepower after filling missing values
print("Median horsepower after filling missing values:", df["horsepower"].median())

Median horsepower after filling missing values: 252.0


### Q7. Sum of weights
- Select all the cars from Asia
- Select only columns vehicle_weight and model_year
- Select the first 7 values
- Get the underlying NumPy array. Let's call it X.
- Compute matrix-matrix multiplication between the transpose of X and X. To get the transpose, use X.T. Let's call the result XTX.
- Invert XTX.
- Create an array y with values [1100, 1300, 800, 900, 1000, 1100, 1200].
- Multiply the inverse of XTX with the transpose of X, and then multiply the result by y. Call the result w.
- What's the sum of all the elements of the result?


In [11]:
# 1, 2, 3
X = df[df["origin"].str.lower() == "asia"][["vehicle_weight", "model_year"]].head(7)

# 4. Convert to NumPy array
X = X.values
print(X)
print("X.shape:", X.shape)

[[4240 1996]
 [4310 1992]
 [4230 1997]
 [4420 1978]
 [3950 2007]
 [4750 2021]
 [4450 2019]]
X.shape: (7, 2)


In [12]:
# 5. Transpose
def matrix_transpose(U: np.ndarray) -> np.ndarray:
    rows = U.shape[0]
    cols = U.shape[1]
    result = np.zeros((cols, rows))

    for i in range(rows):
        for j in range(cols):
            result[j][i] = U[i][j]
    return result


print("Builtin:", X.T)
print()
print("Manual:", matrix_transpose(X))
print()

T = matrix_transpose(X)
print("T:", T)

Builtin: [[4240 4310 4230 4420 3950 4750 4450]
 [1996 1992 1997 1978 2007 2021 2019]]

Manual: [[4240. 4310. 4230. 4420. 3950. 4750. 4450.]
 [1996. 1992. 1997. 1978. 2007. 2021. 2019.]]

T: [[4240. 4310. 4230. 4420. 3950. 4750. 4450.]
 [1996. 1992. 1997. 1978. 2007. 2021. 2019.]]


In [13]:
# Matrix Multiplication
def matrix_multiplication(U: np.ndarray, V: np.ndarray) -> np.ndarray:
    assert U.shape[1] == V.shape[0]

    rows = U.shape[0]  # 3
    cols = V.shape[1]  # 3
    common = U.shape[1]  # 4

    result = np.zeros((rows, cols))

    for i in range(rows):
        for j in range(cols):
            for k in range(common):
                result[i][j] += U[i][k] * V[k][j]

    return result


print("Builtin:", T.dot(X))
print()
print("Manual:", matrix_multiplication(T, X))
print()

XTX = matrix_multiplication(T, X)
print("XTX:", XTX)

Builtin: [[1.3195050e+08 6.0750580e+07]
 [6.0750580e+07 2.8041424e+07]]

Manual: [[1.3195050e+08 6.0750580e+07]
 [6.0750580e+07 2.8041424e+07]]

XTX: [[1.3195050e+08 6.0750580e+07]
 [6.0750580e+07 2.8041424e+07]]


In [14]:
# 6. Invert XTX
XTX_inv = np.linalg.inv(XTX)
print("XTX inverse:", XTX_inv)

XTX inverse: [[ 2.96830537e-06 -6.43071025e-06]
 [-6.43071025e-06  1.39675281e-05]]


In [15]:
# 7, 8
def matrix_vector_multiplication(U: np.ndarray, v: np.ndarray) -> np.ndarray:
    assert U.shape[1] == v.shape[0]

    m = U.shape[0]  # 3 rows
    n = U.shape[1]  # 4 cols

    result = np.zeros(m)

    for i in range(m):
        for j in range(n):
            result[i] = result[i] + U[i][j] * v[j]

    return result


y = np.array([1100, 1300, 800, 900, 1000, 1100, 1200])

w = matrix_multiplication(XTX_inv, T)
w = matrix_vector_multiplication(w, y)
print("w:", w)

w: [0.13644777 0.2327492 ]


In [16]:
# 9. Sum of all elements
answer = w.sum()
print("Sum:", answer)

Sum: 0.3691969690492522
